[Optiver - Trading at the Close](https://www.kaggle.com/competitions/optiver-trading-at-the-close)

[PyTorch DNN](https://www.kaggle.com/code/xquantum/pytorch-dnn)

# Overview
This notebook contains a pipeline for a __PyTorch DNN__ to __predict stock movements__ for the __Optiver Trading Challenge__.

- Customizable Neural Network
- Preprocessing and Normalization of Input Data
- Feature Engineering
- Decaying Learning Rate and Early Stopping
- Hardware Acceleration

## Possible Improvements
- [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html) for Bottlenecks
- Hyperparameter Tuning with [RayTune](https://docs.ray.io/en/latest/tune/index.html) (lr, layers, batchsize)
- Flag filled NaN for near_price and far_price
- Regularisation methods for deeper networks (batchnorm)there is a huge improvement in how you deal with the unknown clips :-D

# DeepL 번역
이 노트북에는 __Optiver Trading Challenge__를 위해 __주식 시세 변동__을 예측하는 __PyTorch DNN__ 파이프라인이 포함되어 있습니다.
- 사용자 정의 가능한 신경망
- 입력 데이터의 전처리 및 정규화
- 특징 공학
- 학습률 감쇠 및 조기 종료
- 하드웨어 가속
## 개선 방안
- 병목 현상 분석을 위한 [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)
- [RayTune](https://docs.ray.io/en/latest/tune/index.html)을 이용한 하이퍼파라미터 튜닝 (학습률, 레이어, 배치 크기)
- near_price 및 far_price에 대해 NaN으로 채우기
- 더 깊은 네트워크를 위한 정규화 방법 (batchnorm) 미지 클립을 처리하는 방식에서 엄청난 개선이 있습니다 :-D

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
plt.tight_layout()
plt.rcParams["figure.figsize"] = (6, 3)
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
# pd.reset_option('display.max_columns')

<Figure size 640x480 with 0 Axes>

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from fastai.tabular.all import *

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


# Data

In [ ]:
df_raw = pd.read_csv("./input/007_optiver-trading-at-the-close/train.csv")
df_raw.isna().sum(axis=0) / len(df_raw)
display(df_raw)

,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,bid_size,ask_price,ask_size,wap,target,time_id,row_id
0,0,0,0,3180602.69,1,0.999812,13380276.64,NaN,NaN,0.999812,60651.50,1.000026,8493.03,1.000000,-3.029704,0,0_0_0
1,1,0,0,166603.91,-1,0.999896,1642214.25,NaN,NaN,0.999896,3233.04,1.000660,20605.09,1.000000,-5.519986,0,0_0_1
2,2,0,0,302879.87,-1,0.999561,1819368.03,NaN,NaN,0.999403,37956.00,1.000298,18995.00,1.000000,-8.389950,0,0_0_2
3,3,0,0,11917682.27,-1,1.000171,18389745.62,NaN,NaN,0.999999,2324.90,1.000214,479032.40,1.000000,-4.010200,0,0_0_3
4,4,0,0,447549.96,-1,0.999532,17860614.95,NaN,NaN,0.999394,16485.54,1.000016,434.10,1.000000,-7.349849,0,0_0_4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5237975,195,480,540,2440722.89,-1,1.000317,28280361.74,0.999734,0.999734,1.000317,32257.04,1.000434,319862.40,1.000328,2.310276,26454,480_540_195
5237976,196,480,540,349510.47,-1,1.000643,9187699.11,1.000129,1.000386,1.000643,205108.40,1.000900,93393.07,1.000819,-8.220077,26454,480_540_196
5237977,197,480,540,0.00,0,0.995789,12725436.10,0.995789,0.995789,0.995789,16790.66,0.995883,180038.32,0.995797,1.169443,26454,480_540_197
5237978,198,480,540,1000898.84,1,0.999210,94773271.05,0.999210,0.999210,0.998970,125631.72,0.999210,669893.00,0.999008,-1.540184,26454,480_540_198


https://www.kaggle.com/code/xquantum/pytorch-dnn